In [1]:
# added top-p and top-k filtering in generate function
# set vocab_size in config.py
# MHA with KV cache + RoPE + PyTorch SDPA.
# This traditional implementation is easier to understand, and still efficient in practice.
# GQA and MLA is a great way for long-text inference with reduced KV cache size,
# but both comes with slight loss increase and no efficiency merits during training phase.
# KV cache does not help training speed. Codebase will be simpler without it.
# KV cache supports multi-turn continuation by RoPE with position offset.
# No Dropout. Dataset is large enough and regularization is not necessary.

import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.embedding_dim)
        # keep embedding in default dtype (autocast will handle bf16 when enabled)

    def forward(self, input_indices):
        return self.token_embedding_table(input_indices)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, rope_theta=1e6):
        super().__init__()

        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2) / dim))
        position_index = torch.arange(max_seq_len)
        frequency_matrix = torch.einsum('i,j->ij', position_index, inv_freq)

        cosine = torch.cos(frequency_matrix)[None, None, :, :]
        sine = torch.sin(frequency_matrix)[None, None, :, :]

        self.register_buffer("cos_cached", cosine, persistent=False)
        self.register_buffer("sin_cached", sine, persistent=False)

    def apply_rotary_emb(self, x, position_offset=0):
        sequence_length = x.size(2)

        cosine = self.cos_cached[:, :, position_offset:position_offset + sequence_length, :]
        sine = self.sin_cached[:, :, position_offset:position_offset + sequence_length, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        rotated_even = x_even * cosine - x_odd * sine
        rotated_odd = x_odd * cosine + x_even * sine

        rotated = torch.empty_like(x)
        rotated[..., 0::2] = rotated_even
        rotated[..., 1::2] = rotated_odd

        return rotated

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_heads = config.num_attention_heads
        self.embed_dim = config.embedding_dim
        self.head_dim = self.embed_dim // self.num_heads

        # QKV projection
        self.query_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.key_fc   = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.value_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)

        # Rotary Positional Embedding (RoPE)
        self.rotary_emb = RotaryEmbedding(
            dim=self.head_dim,
            max_seq_len=config.max_sequence_length,
            rope_theta=config.rope_theta
        )

        self.output_projection = nn.Linear(self.embed_dim, self.embed_dim)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(
                config.max_sequence_length,
                config.max_sequence_length,
                dtype=torch.bool
            )),
            persistent=False
        )

        # KV cache
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.current_pos = 0

    # --------------------------------------------------
    # router
    # --------------------------------------------------
    def forward(self, x, use_cache=False):
        input_len = x.size(1)
        if use_cache is False:
            return self.forward_no_cache(x)
        elif use_cache is True and input_len > 1:
            return self.forward_prefill(x)
        elif use_cache is True and input_len == 1: # Hi scenario also starts with T==1
            return self.forward_cached_decoding(x)
        else:
            raise RuntimeError("Unexpected condition in MultiHeadAttention forward")

    # --------------------------------------------------
    # (1) no cache : training 
    # --------------------------------------------------
    def forward_no_cache(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # RoPE : offset = 0
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=0)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=0)

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (2) prefill : initialize KV cache
    # --------------------------------------------------
    def forward_prefill(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # init cache
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        # RoPE : offset = current_pos (supports multi-turn continuation)
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        # prevent overflow
        if self.current_pos + T > self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        self.cache_k[:, :, self.current_pos:self.current_pos + T, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + T, :] = V

        K = self.cache_k[:, :, :self.current_pos + T, :]
        V = self.cache_v[:, :, :self.current_pos + T, :]

        attn_mask = self.causal_mask[
            self.current_pos : self.current_pos + T,
            : self.current_pos + T
        ]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            is_causal=False
        )

        self.current_pos += T

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (3) decode : cached decoding (1 token)
    # --------------------------------------------------
    def forward_cached_decoding(self, x):
        B, T, C = x.shape
        assert T == 1, "cached decoding expects T==1"

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)

        # This is not usually needed since prefill should have initialized the cache.
        # Just in case for "Hi" scenario, which starts with single token input.
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        if self.current_pos + 1 >= self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        # RoPE : offset = current_pos
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        self.cache_k[:, :, self.current_pos:self.current_pos + 1, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + 1, :] = V

        K = self.cache_k[:, :, :self.current_pos + 1, :]
        V = self.cache_v[:, :, :self.current_pos + 1, :]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=False
        )

        self.current_pos += 1

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.current_pos = 0



class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()    
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim, bias=False),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)
        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)


    def forward(self, input_tensor, use_cache=False):
        normed_input = self.layer_norm1(input_tensor)
        attention_output = self.multihead_attention(normed_input, use_cache=use_cache)
        residual_attention = attention_output + input_tensor
        normed_attention = self.layer_norm2(residual_attention)
        feedforward_output = self.feed_forward(normed_attention)
        final_output = feedforward_output + residual_attention
        return final_output


class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        self.vocab_projection = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)

    def forward(self, transformer_block_output):
        x = transformer_block_output
        normalized_output = self.output_norm(x)
        vocab_logits = self.vocab_projection(normalized_output)
        return vocab_logits


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_layer = TokenEmbedding(config=config)
        self.blocks = nn.ModuleList([TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(config=config)
        self.criterion = nn.CrossEntropyLoss()


    def forward(self, input_indices, target_indices, use_cache=False):
        token_embeddings = self.token_embedding_layer.forward(input_indices)

        x = token_embeddings
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        logits = self.vocab_projection(x)

        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits_flat = logits.view(batch_size * token_len, vocab_size)
        targets_flat = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits_flat, targets_flat)
        return logits, loss


    def generate(self,
        input_indices,
        max_new_tokens,
        temperature=1.0,
        use_cache=True,
        reset_cache=False,
        top_k=None,      # ### NEW ###
        top_p=None,      # ### NEW ###
    ):
        self.eval()

        if reset_cache:
            for block in self.blocks:
                block.multihead_attention.reset_cache()

        next_token = None

        for i in range(max_new_tokens):
            if use_cache:
                if i == 0:
                    logits, _ = self.forward(input_indices, None, use_cache=True)
                else:
                    logits, _ = self.forward(next_token, None, use_cache=True)
            else:
                logits, _ = self.forward(input_indices, None, use_cache=False)

            """ DELETE
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            """

            ### NEW ###
            last_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_k = min(top_k, last_logits.size(-1))
                values, _ = torch.topk(last_logits, top_k)
                min_value = values[:, -1].unsqueeze(-1)
                last_logits = torch.where(
                    last_logits < min_value,
                    torch.full_like(last_logits, float("-inf")),
                    last_logits,
                )

            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(last_logits, descending=True)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                sorted_mask = cumulative_probs > top_p
                sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
                sorted_mask[..., 0] = False

                sorted_logits = torch.where(
                    sorted_mask,
                    torch.full_like(sorted_logits, float("-inf")),
                    sorted_logits,
                )

                last_logits = torch.zeros_like(last_logits).scatter(
                    -1, sorted_indices, sorted_logits
                )

            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            ### NEW ###

            yield int(next_token.item())
            input_indices = torch.cat((input_indices, next_token), dim=1)

In [2]:
class Config:
    embedding_dim: int = 2560
    hidden_dim: int = 10240
    num_attention_heads: int = 20
    layer_count: int = 30
    rope_theta: float = 1_000_000.0
    vocab_size: int = 50257
    max_sequence_length: int = 2048

In [3]:
config = Config()
model = GPT(config)

In [4]:
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

In [5]:
import random
RANDOM_SEED = 1337
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [6]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO_ID = "HayatoHongo/AIkenSGTv1"
MODEL_FILENAME = "prompt_mask_instruction_tuned_model_epoch_2_lr_1e-04_gkentei_text.safetensors"
model_path = hf_hub_download(repo_id=REPO_ID, filename=MODEL_FILENAME)
print("Model:", MODEL_FILENAME)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Model: prompt_mask_instruction_tuned_model_epoch_2_lr_1e-04_gkentei_text.safetensors


In [7]:
import tiktoken
from safetensors.torch import load_file

tokenizer = tiktoken.get_encoding("gpt2")
state_dict = load_file(model_path, device="cpu")
model.load_state_dict(state_dict)

<All keys matched successfully>

In [8]:
model = model.to(device)
model.eval()

GPT(
  (token_embedding_layer): TokenEmbedding(
    (token_embedding_table): Embedding(50257, 2560)
  )
  (blocks): ModuleList(
    (0-29): 30 x TransformerBlock(
      (layer_norm1): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
      (layer_norm2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
      (multihead_attention): MultiHeadAttention(
        (query_fc): Linear(in_features=2560, out_features=2560, bias=False)
        (key_fc): Linear(in_features=2560, out_features=2560, bias=False)
        (value_fc): Linear(in_features=2560, out_features=2560, bias=False)
        (rotary_emb): RotaryEmbedding()
        (output_projection): Linear(in_features=2560, out_features=2560, bias=True)
      )
      (feed_forward): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=2560, out_features=10240, bias=False)
          (1): ReLU()
          (2): Linear(in_features=10240, out_features=2560, bias=False)
        )
      )
    )
  )
  (vocab_projection): 

## test セット評価

In [9]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="HayatoHongo/AIkenSGTv1",
    repo_type="model",
    filename="mock_questions_v2_direct_fewshot_full_sorted_test.jsonl",
    local_dir=".",
)

'/content/mock_questions_v2_direct_fewshot_full_sorted_test.jsonl'

In [10]:
from google.colab import files
from pathlib import Path
import json

TEST_PATH = Path("/content/mock_questions_v2_direct_fewshot_full_sorted_test.jsonl")
with TEST_PATH.open(encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]


In [ ]:
DEV_PATH = Path("/content/mock_questions_v2_direct_fewshot_full_sorted_dev.jsonl")
with DEV_PATH.open(encoding="utf-8") as f:
    dev_data = [json.loads(line) for line in f]

FEW_SHOT_COUNT = 5  # 0〜5
few_shot_prompt = "".join(
    f"<USER>{item['prompt']}<ASSISTANT>{item['response']}<|endoftext|>"
    for item in dev_data[:FEW_SHOT_COUNT]
)


In [35]:
import pandas as pd
from IPython.display import display

results = []
for i, item in enumerate(test_data, start=1):
    prompt = few_shot_prompt + "<USER>" + item["prompt"] + "<ASSISTANT>"
    input_ids = torch.tensor([tokenizer.encode(prompt, allowed_special="all")], device=device)
    output_ids = []

    with torch.inference_mode():
        for token in model.generate(input_ids, max_new_tokens=512, top_k=None, temperature = 0.5, reset_cache=True):
            if token == tokenizer.eot_token:
                break
            output_ids.append(token)

    raw_output = tokenizer.decode(output_ids)
    results.append({
        "row": i,
        "prompt": item["prompt"],
        "expected_response": item["response"],
        "raw_output": raw_output,
        "exact_match": raw_output == item["response"],
    })

results = pd.DataFrame(results)
OUTPUT_PATH = "/content/gkentei_test_results_eval1temp01.csv"
results.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"完全一致: {results['exact_match'].sum()}/{len(results)}")
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
display(results)

完全一致: 52/145


,row,prompt,expected_response,raw_output,exact_match
0,1,学習に利用できるデータが限られている状況で、複数の分類モデルの候補を比較することになった。一度のデータ分割によって評価結果が左右されにくくなる方法として、最も適切なものを1つ選べ。\nA. k-分割交差検証を用いる\nB. ホールドアウト法を用いる\nC. 学習曲線を用いる\nD. 混同行列を用いる,\boxed{k-分割交差検証を用いる},\boxed{混同行列を用いる},False
1,2,データ分析の試行錯誤では、コードを部分ごとに実行して結果を確認し、その過程の説明も同じ文書に残したい場合がある。この用途に適した（あ）では、コードを（い）に分けて記述し、見出しや説明文を（う）として記録できる。空欄の組み合わせとして、最も適切なものを1つ選べ。\nA. （あ）統合開発環境、（い）セル、（う）Markdown\nB. （あ）Jupyter Notebook、（い）セル、（う）Markdown\nC. （あ）統合開発環境、（い）ソースファイル、（う）実行ログ\nD. （あ）Jupyter Notebook、（い）ソースファイル、（う）実行ログ,\boxed{（あ）Jupyter Notebook、（い）セル、（う）Markdown},\boxed{（あ）統合開発環境、（い）ソースファイル、（う）実行ログ},False
2,3,ある入力に対する正解側の確率分布をP、モデルの予測分布をQとする。Pを基準としたQのずれを評価する指標であるKLダイバージェンスについての説明として、最も適切なものを1つ選べ。\nA. Pを基準にKL(Q||P)を計算し、分布の順序を指定して評価する\nB. Pを基準にKL(P||Q)を計算し、分布の順序を指定して評価する\nC. PとQの各確率の差を平均し、分布の順序に依存しない値で評価する\nD. PとQで一致する確率だけを集計し、分布のずれを評価する,\boxed{Pを基準にKL(P||Q)を計算し、分布の順序を指定して評価する},\boxed{PとQで一致する確率だけを集計し、分布のずれを評価する},False
3,4,同じ条件で部品を1個無作為に選び、その重量を測定する。測定値をXとするとき、Xを確率変数として扱うことの説明として最も適切なものを1つ選べ。\nA. 選ばれた部品の結果をまとめ、重量の分布全体を表す変数である\nB. 選ばれた部品の結果に応じて値が定まり、重量を数値で表す変数である\nC. 選ばれた部品の結果によらず、全体の平均重量を表す変数である\nD. 選ばれた部品の結果ごとに、その結果の発生確率を表す変数である,\boxed{選ばれた部品の結果に応じて値が定まり、重量を数値で表す変数である},\boxed{選ばれた部品の結果ごとに、その結果の発生確率を表す変数である},False
4,5,音声認識の前処理として、人間の聴覚特性を考慮して音声のスペクトルを表現する特徴量を用いたい。この目的に最も適切なものを1つ選べ。\nA. 基本周波数を用いて、音声の音高の時間変化を特徴として表す\nB. スペクトログラムを用いて、音声スペクトルを時間と線形周波数の特徴として表す\nC. MFCCを用いて、音声スペクトルをメル尺度に基づく特徴量として表す\nD. 音声波形を用いて、音声の時間ごとの振幅を特徴として表す,\boxed{MFCCを用いて、音声スペクトルをメル尺度に基づく特徴量として表す},\boxed{スペクトログラムを用いて、音声スペクトルを時間と線形周波数の特徴として表す},False
5,6,ある二値分類モデルの適合率が0.75、再現率が0.50のとき、このモデルのF値として正しいものを1つ選べ。\nA. 0.600\nB. 0.625\nC. 0.500\nD. 0.375,\boxed{0.600},\boxed{0.600},True
6,7,会議中の発話を入力として、話者を特定するのではなく、発言内容を文字列として出力する技術を（_____）という。空欄に最もよく当てはまる選択肢を1つ選べ。\nA. 音源分離\nB. 音声合成\nC. 音声認識\nD. 話者認識,\boxed{音声認識},\boxed{話者認識},False
7,8,音声合成では、過去に生成した波形の情報を利用しながら、音声波形を逐次的に生成するニューラルネットワークを（_____）という。空欄に最もよく当てはまる選択肢を1つ選べ。\nA. WaveNet\nB. MelGAN\nC. DeepSpeech\nD. Tacotron,\boxed{WaveNet},\boxed{DeepSpeech},False
8,9,決定木を用いた故障判定モデルで、学習データの違いによる予測結果の不安定さを抑えたい。複数のモデルを利用して予測を安定させる方法のうち、最も適切なものを1つ選べ。\nA. バギングを用い、元の学習データから異なるデータセットを再標本化して複数のモデルを学習し、予測結果を統合する\nB. ブースティングを用い、前のモデルが誤ったデータを重視しながらモデルを順番に学習し、予測結果を統合する\nC. ドロップアウトを用い、1つのニューラルネットワークで学習時に一部のユニットを無効化する\nD. スタッキングを用い、複数のモデルの予測結果を別のモデルに入力して最終的な予測を得る,\boxed{バギングを用い、元の学習データから異なるデータセットを再標本化して複数のモデルを学習し、予測結果を統合する},\boxed{ドロップアウトを用い、1つのニューラルネットワークで学習時に一部のユニットを無効化する},False
9,10,画像認識モデルの訓練用画像に対し、撮影条件の違いに対する対応力を高めるためのデータ拡張を行う。画像全体の明暗の差を調整する操作を（あ）、画像全体の明るさを調整する操作を（い）、画像の向きを変える操作を（う）という。空欄の組合せとして、最も適切なものを1つ選べ。\nA. （あ）Contrast、（い）Rotation、（う）Brightness\nB. （あ）Brightness、（い）Contrast、（う）Rotation\nC. （あ）Rotation、（い）Brightness、（う）Contrast\nD. （あ）Contrast、（い）Brightness、（う）Rotation,\boxed{（あ）Contrast、（い）Brightness、（う）Rotation},\boxed{（あ）Rotation、（い）Brightness、（う）Contrast},False


In [36]:
files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>